# 10 · Rules as data: expressions

**The problem.** Marketing runs promotions with eligibility rules: "65 or older, with an email address on file",
"between 18 and 65", "a VIP, or someone spending over a limit". An admin screen edits the rules, the back end (Python)
applies them to contacts, and the web front end (TypeScript) shows who qualifies. So a rule has to be *stored* with the
rest of the data, *sent* between programs, and *evaluated* the same way by both.

A TypeScript function can do none of that: it can't be saved, and Python can't run it. An **expression** can. It's
data with a schema, like everything else in this framework. This case study follows an expression through its life:
building it, evaluating it, saving and loading it, and walking it to analyze or rewrite it.

In TypeScript you write an expression with **terms**: method calls that build the data. The Python tutorial writes the
same rules as lambdas, which `Expressions.from_` reads from their source. A JavaScript function has no Python source to
read, so TypeScript has no `from_`; the data is identical either way.

In [1]:
import { Evaluators, Expressions, JSON as Json, Proxies, Schemas } from "../src/Framework/index.js";

const Contact = new Schemas.OfObject.Builder()
  .properties(
    (p) => p.name("name").of((t) => t.as_native(String)),
    (p) => p.name("age").of((t) => t.as_native(BigInt)),
    (p) => p.name("email").of((t) => t.as_native(String)),
    (p) => p.name("vip").of((t) => t.as_native(Boolean)),
  )
  .create();
Proxies.register("Contact", Contact);
const NewContact = Proxies.Builders.Contact;

const ada = NewContact().name("Ada").age(70n).email("ada@example.com").create();
const bob = NewContact().name("Bob").age(40n).vip(true).create();
const cy = NewContact().name("Cy").email("cy@example.com").create(); // no age on file
const people = [ada, bob, cy];

/** An expression as function-call text. Section 4 explains how it works. */
function show(expression: Expressions.OfAny.Spec): string {
  const e = Expressions.OfAny.resolve(expression);
  if (e instanceof Expressions.OfLiteral.Data) return typeof e.value === "string" ? `'${e.value}'` : String(e.value);
  if (e instanceof Expressions.OfVariable.Data) return String(e.name);
  if (e instanceof Expressions.OfLet.Data) return `let ${e.name} = ${show(e.value as any)} in ${show(e.body as any)}`;
  return `${e.name}(${e.arguments.map((argument) => show(argument as any)).join(", ")})`;
}

## 1. Building an expression

A variable is a term. `.name` on a term reads a property (`get`), and methods build the operations of the core
vocabulary:

In [2]:
const contact = Expressions.variable("contact");
const senior = contact.age.ge(65n).and_(contact.has("email"));
console.log(show(senior));

and(ge(get(contact, 'age'), 65), has(contact, 'email'))


`contact.age` reads a property, `.ge` is `>=`, `.and_` is `and`, and `.has("email")` asks whether a property is
present. The methods are `eq`, `ne`, `lt`, `le`, `gt`, `ge`, `and_`, `or_`, `not_`, `implies`, `add`, `sub`, `mul`,
`neg`, `has` and `get` (for a property whose name is also a method's). The trailing underscores keep `and`, `or` and
`not` the same in both languages, where Python reserves them.

A term takes values from your program when it's built: `MIN_SPEND` below becomes a literal, and the expression doesn't
refer back to the constant. Another variable is another parameter of the rule:

In [3]:
const MIN_SPEND = 1000n;
const spend = Expressions.variable("spend");
const workingAge = contact.age.ge(18n).and_(contact.age.lt(65n));
const bigSpender = contact.vip.or_(spend.gt(MIN_SPEND));
console.log(show(workingAge));
console.log(show(bigSpender));

and(ge(get(contact, 'age'), 18), lt(get(contact, 'age'), 65))


or(get(contact, 'vip'), gt(spend, 1000))


To name an intermediate value, use a **let**: the name is bound to the value inside the body, and nowhere else.

In [4]:
const age = Expressions.variable("age");
const sixties = Expressions.let_("age", contact.age, age.ge(60n).and_(age.lt(70n)));
console.log(show(sixties));

let age = get(contact, 'age') in and(ge(age, 60), lt(age, 70))


Terms accept any operation name, since the vocabulary is open to extensions. Whether a rule sticks to the core
vocabulary is checked by `validate()` (section 2), and evaluating an operation outside it throws:

In [5]:
const tenths = Expressions.operation("div", contact.age, 10n);
console.log(tenths.data.validate({ bound: ["contact"], core: true }));

[ "'div' is not a core operation" ]


## 2. Evaluating an expression

`Evaluators.OfAny(expression, scope)` computes the value, with the variables in `scope` bound:

In [6]:
for (const person of people) console.log(person.name, Evaluators.OfAny(senior, { contact: person }));

Ada true


Bob false


Cy null


Ada qualifies and Bob doesn't. Cy's answer is `null`: her age isn't on file, so whether she is 65 or older is
**unknown**. A plain function reading `cy.age` would have thrown `AttributeError`. An expression answers instead, and
says it doesn't know.

Unknowns combine with *three-valued logic*: `false and unknown` is `false`, `true or unknown` is `true`, and otherwise
unknown stays unknown. So a rule can still decide when enough is known:

In [7]:
const reachable = contact.has("email").or_(contact.age.ge(65n));
console.log(people.map((person) => Evaluators.OfAny(reachable, { contact: person })));

[ true, false, true ]


A rule with two parameters needs both bound. Bob is a VIP, so what he spends doesn't matter; Ada's VIP flag is
absent, so her answer depends on the spend:

In [8]:
console.log(Evaluators.OfAny(bigSpender, { contact: bob, spend: 50n }));
console.log(Evaluators.OfAny(bigSpender, { contact: ada, spend: 1500n }));
console.log(Evaluators.OfAny(bigSpender, { contact: ada, spend: 50n }));

true


true


null


**No coercion.** As everywhere in this framework (case study 7), an integer (`bigint`) and a float (`number`) are
different kinds of value. Comparing them is unknown rather than a guess, which is also what a Python program comparing
an `int` with a `float` sees:

In [9]:
console.log(Evaluators.OfAny(contact.age.ge(65), { contact: ada }));

null


Rules can be checked before they run. `validate()` reports what evaluation would raise: variables nothing binds,
wrong numbers of arguments, and, with `core: true`, operations outside the core vocabulary:

In [10]:
console.log(bigSpender.data.validate({ bound: ["contact", "spend"], core: true }));
console.log(bigSpender.data.validate({ bound: ["contact"] }));

[]


[ "argument 1: argument 0: variable 'spend' is not bound" ]


## 3. Saving and loading an expression

An expression's data is `Visitable`, and every kind has a meta-schema, so it saves like any object graph (case study
6):

In [11]:
const text = Json.ToJSON.Reachable(Expressions.OfOperation.Schema, senior.data);
console.log(text);

{"root": "s0", "objects": {"s0": {"kind": "operation", "name": "and", "arguments": [{"argument": {"$ref": "s1", "$schema": "Expressions.OfOperation"}, "index": 0}, {"argument": {"$ref": "s2", "$schema": "Expressions.OfOperation"}, "index": 1}]}, "s1": {"kind": "operation", "name": "ge", "arguments": [{"argument": {"$ref": "s3", "$schema": "Expressions.OfOperation"}, "index": 0}, {"argument": {"$ref": "s4", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s2": {"kind": "operation", "name": "has", "arguments": [{"argument": {"$ref": "s5", "$schema": "Expressions.OfVariable"}, "index": 0}, {"argument": {"$ref": "s6", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s3": {"kind": "operation", "name": "get", "arguments": [{"argument": {"$ref": "s5", "$schema": "Expressions.OfVariable"}, "index": 0}, {"argument": {"$ref": "s7", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s4": {"kind": "literal", "int": 65}, "s5": {"kind": "variable", "name": "contact"}, "s6": {"kind": "liter

Each object carries a `kind` tag, arguments are entries with an `index`, and the variable `contact`, used twice, is
written once. This is exactly the text the Python tutorial produces for the same rule, so either program can load
what the other saved. Loading it here uses `Expressions.Builders`, which build expression data rather than proxies:

In [12]:
const loaded = Json.FromJSON(Expressions.Builders).Reachable(Expressions.OfOperation.Schema, text) as Expressions.OfAny.Data;
console.log(show(loaded));
console.log(Json.ToJSON.Reachable(Expressions.OfOperation.Schema, loaded) === text);
console.log(people.map((person) => Evaluators.OfAny(loaded, { contact: person })));

and(ge(get(contact, 'age'), 65), has(contact, 'email'))


true


[ true, false, null ]


## 4. Traversing an expression for analysis

An expression is a small tree of four kinds of data, each with plain fields:

| Kind | Fields |
|---|---|
| `Expressions.OfLiteral.Data` | `value` |
| `Expressions.OfVariable.Data` | `name` |
| `Expressions.OfOperation.Data` | `name`, `arguments` |
| `Expressions.OfLet.Data` | `name`, `value`, `body` |

`show`, above, is such a traversal. Here is a more useful one: which properties does a rule read? The admin screen
can use it to load only those fields, or to warn when a rule reads a property the schema doesn't have.

In [13]:
/** The "variable.property" pairs an expression reads with get or has. */
function reads(expression: Expressions.OfAny.Spec): Set<string> {
  const e = Expressions.OfAny.resolve(expression);
  if (e instanceof Expressions.OfLet.Data) return new Set([...reads(e.value as any), ...reads(e.body as any)]);
  if (!(e instanceof Expressions.OfOperation.Data)) return new Set();
  const found = new Set(e.arguments.flatMap((argument) => [...reads(argument as any)]));
  const [subject, name] = e.arguments;
  if ((e.name === "get" || e.name === "has") && subject instanceof Expressions.OfVariable.Data) {
    found.add(`${subject.name}.${(name as Expressions.OfLiteral.Data).value}`);
  }
  return found;
}

for (const rule of [senior, bigSpender, sixties]) console.log(show(rule), "->", [...reads(rule)].sort());

and(ge(get(contact, 'age'), 65), has(contact, 'email')) -> [ "contact.age", "contact.email" ]


or(get(contact, 'vip'), gt(spend, 1000)) -> [ "contact.vip" ]


let age = get(contact, 'age') in and(ge(age, 60), lt(age, 70)) -> [ "contact.age" ]


Scoping matters in a traversal. A variable bound by a let isn't a parameter of the rule; `freeVariables` keeps
track of the names each let binds:

In [14]:
/** The variables an expression needs bound by whoever evaluates it. */
function freeVariables(expression: Expressions.OfAny.Spec, bound: ReadonlySet<string> = new Set()): Set<string> {
  const e = Expressions.OfAny.resolve(expression);
  if (e instanceof Expressions.OfVariable.Data) return bound.has(e.name as string) ? new Set() : new Set([e.name as string]);
  if (e instanceof Expressions.OfLet.Data) {
    const inner = new Set([...bound, e.name as string]);
    return new Set([...freeVariables(e.value as any, bound), ...freeVariables(e.body as any, inner)]);
  }
  if (e instanceof Expressions.OfOperation.Data) {
    return new Set(e.arguments.flatMap((argument) => [...freeVariables(argument as any, bound)]));
  }
  return new Set();
}

console.log([...freeVariables(bigSpender)].sort());
console.log([...freeVariables(sixties)].sort());

[ "contact", "spend" ]


[ "contact" ]


`validate({ bound })` makes the same check. Writing it yourself shows the pattern every analysis follows: one case
per kind, recursing into arguments, lets' values and lets' bodies.

## 5. Traversing an expression for substitution

A rule can be written once as a *template* and specialized later. Here the age limit is a parameter; a campaign fixes
it by replacing the variable with a literal. Substitution rebuilds the tree with builders, so the template itself is
untouched:

In [15]:
/** A copy of `expression` with the free variables named in `values` replaced by literals. */
function substitute(expression: Expressions.OfAny.Spec, values: Record<string, bigint | number | string | boolean>,
  bound: ReadonlySet<string> = new Set()): Expressions.OfAny.Data {
  const e = Expressions.OfAny.resolve(expression);
  if (e instanceof Expressions.OfVariable.Data) {
    const name = e.name as string;
    return name in values && !bound.has(name) ? Expressions.OfLiteral.resolve(values[name]) : e;
  }
  if (e instanceof Expressions.OfLet.Data) {
    return new Expressions.OfLet.Builder().name(e.name as string).value(substitute(e.value as any, values, bound))
      .body(substitute(e.body as any, values, new Set([...bound, e.name as string]))).create();
  }
  if (e instanceof Expressions.OfOperation.Data) {
    return new Expressions.OfOperation.Builder().name(e.name as string)
      .arguments(...e.arguments.map((argument) => substitute(argument as any, values, bound))).create();
  }
  return e;
}

const limit = Expressions.variable("limit");
const atLeast = contact.age.ge(limit);
const seniorCampaign = substitute(atLeast, { limit: 65n });
console.log(show(atLeast), "->", show(seniorCampaign));
console.log([...freeVariables(seniorCampaign)], Evaluators.OfAny(seniorCampaign, { contact: ada }));

ge(get(contact, 'age'), limit) -> ge(get(contact, 'age'), 65)


[ "contact" ] true


The `bound` set stops substitution at a let that binds the same name, just as evaluation would:

In [16]:
const shadowed = Expressions.let_("limit", 18n, contact.age.ge(limit));
console.log(show(substitute(shadowed, { limit: 65n })));

let limit = 18 in ge(get(contact, 'age'), limit)


Parts that don't change, such as literals and variables that aren't replaced, are shared between the template
and the copy rather than copied. Expressions are never changed in place, so sharing is safe.

## What's next

Expressions are what union discriminators are made of (case study 9): each branch's predicate is evaluated with
`this` bound to the value, and the first that is `true` chooses the branch. Choosing branches that way, and the
collection operations (`count`, `in`, `all`, `any`) that constraints like "at least one phone" need, are designed but
not built yet.

## Appendix: building expressions with builders

Terms are the everyday notation. Underneath, each kind has a builder, like every other element of the framework,
finalized by `create()`, `clone()` or `update()`. Builders are the lowest level; section 5 used them because they take
data apart and put it together. Built by hand, the senior rule is:

In [17]:
const ageOf = new Expressions.OfOperation.Builder().name("get").arguments(contact, "age").create();
const built = new Expressions.OfOperation.Builder().name("and").arguments(
  new Expressions.OfOperation.Builder().name("ge").arguments(ageOf, 65n).create(),
  new Expressions.OfOperation.Builder().name("has").arguments(contact, "email").create(),
).create();
console.log(show(built));
console.log(Json.ToJSON.Reachable(Expressions.OfOperation.Schema, built) === text);

and(ge(get(contact, 'age'), 65), has(contact, 'email'))


true


The JSON is identical to the one the terms produced in section 3: both notations build the same data.
`Expressions.literal`, `Expressions.let_` and `Expressions.operation` (for operations outside the core vocabulary)
start other terms. Use terms to write and assemble rules, and builders when you take expressions apart or rebuild
them.

## The whole tour

| Case study | Problem | Idea |
|---|---|---|
| 1 | a contact card | schemas are values; builders; `create`/`update`/`clone`; absent is not `undefined` |
| 2 | an address book | relations instead of arrays; untyped links; entries are shared facts |
| 3 | moving house | editing entries through builders; what `update()` replaces |
| 4 | a family tree | self-relations, cycles, `Reachable.of`; unambiguous inference |
| 5 | a price list | `unique(...)` cardinality; validation on demand |
| 6 | saving and loading | snapshots, symbols, references with schema names |
| 7 | JSON and YAML | strict text; the Norway problem; no coercion |
| 8 | tools for every schema | walking schemas; writing a visitor |
| 9 | evolving schemas | optional fields; versions; directories; variants |
| 10 | rules as data | expressions: building, evaluating, saving, analyzing and rewriting them |

The through-line: **describe data once, as data, and let every tool read that description.** Most of the ways this
framework differs from the usual approach (no arrays, no mandatory fields, no automatic validation, no type
coercion, no guessing) are there so that description means the same thing to every program that reads it, in
TypeScript, in Python, and in whatever comes next. Rules
are no exception: an expression is data too.